# See datasets in /data/cpanourg/2-hdvc/data

Load and inspect the vector datasets: bigann, deep1b, gist, msmarco, openai.

In [1]:
import sys
from pathlib import Path

import numpy as np

# Add project root for imports (works from project root or scripts/dataset/)
def _find_project_root():
    p = Path.cwd()
    for _ in range(5):
        if (p / "src").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_dataset, read_fvecs, read_fvecs_cnt

DATA_ROOT = Path("/data/cpanourg/2-hdvc/data")


def read_bin(path, dim, max_vectors=None):
    """Read .bin float32 vectors with optional limit."""
    if max_vectors:
        data = np.fromfile(path, dtype=np.float32, count=max_vectors * dim)
        return data.reshape(-1, dim).astype(np.float32)
    return np.fromfile(path, dtype=np.float32).reshape(-1, dim).astype(np.float32)

In [2]:
# bigann (SIFT1M subset, .bvecs format)
def read_bvecs(path, max_vectors=None):
    """Read .bvecs (per vector: 4-byte dim + dim bytes uint8)."""
    with open(path, "rb") as f:
        dim = int(np.frombuffer(f.read(4), dtype=np.int32)[0])
        vec_size = 4 + dim
        data = []
        n = 0
        while True:
            to_read = min(10000, max_vectors - n) if max_vectors else 10000
            if max_vectors and n >= max_vectors:
                break
            chunk = f.read(vec_size * to_read)
            if len(chunk) < vec_size:
                break
            k = len(chunk) // vec_size
            arr = np.frombuffer(chunk[: k * vec_size], dtype=np.uint8).reshape(k, vec_size)[:, 4:].astype(np.float32)
            data.append(arr)
            n += k
        if not data:
            return np.zeros((0, dim), dtype=np.float32)
        out = np.vstack(data)
        return out[:max_vectors] if max_vectors else out



In [3]:
MAX_VECTORS = None  # Limit per dataset for quick inspection (set to None to load all)
DEEP1B_DIR = DATA_ROOT / "deep1b/dataset/fvecs"
db_deep1b = read_fvecs_cnt(str(DEEP1B_DIR / "test_1m.fvecs"), MAX_VECTORS or 1_000_000)
learn_deep1b = read_fvecs_cnt(str(DEEP1B_DIR / "learn_100m.fvecs"), MAX_VECTORS or 100_000_000)
qr_deep1b = read_fvecs_cnt(str(DEEP1B_DIR / "query_10k.fvecs"), 10_000)
print(f"deep1b: base {db_deep1b.shape}, learn {learn_deep1b.shape}, query {qr_deep1b.shape}")

Reading File - /data/cpanourg/2-hdvc/data/deep1b/dataset/fvecs/test_1m.fvecs, with 1000000 vectors:(1000000, 96)
Reading File - /data/cpanourg/2-hdvc/data/deep1b/dataset/fvecs/learn_100m.fvecs, with 100000000 vectors:(100000000, 96)
Reading File - /data/cpanourg/2-hdvc/data/deep1b/dataset/fvecs/query_10k.fvecs, with 10000 vectors:(10000, 96)
deep1b: base (1000000, 96), learn (100000000, 96), query (10000, 96)


In [5]:
MAX_VECTORS = 1_000_000  # Limit per dataset for quick inspection (set to None to load all)
BIGANN_DIR = DATA_ROOT / "bigann/SIFT1M"
db_bigann = read_bvecs(BIGANN_DIR / "bigann_base.bvecs", MAX_VECTORS)
learn_bigann = read_bvecs(BIGANN_DIR / "bigann_learn.bvecs", MAX_VECTORS)
qr_bigann = read_bvecs(BIGANN_DIR / "bigann_query.bvecs", 10_000)
print(f"bigann: base {db_bigann.shape}, learn {learn_bigann.shape}, query {qr_bigann.shape}")

bigann: base (1000000, 128), learn (1000000, 128), query (9999, 128)


In [6]:
# gist (.fvecs)
MAX_VECTORS = None
db_gist = read_fvecs_cnt(str(DATA_ROOT / "gist/gist_base.fvecs"), MAX_VECTORS or 1_000_000)
db_gist = read_fvecs_cnt(str(DATA_ROOT / "gist/gist_learn.fvecs"), MAX_VECTORS or 500_000)
qr_gist = read_fvecs_cnt(str(DATA_ROOT / "gist/gist_query.fvecs"), 1_000)  # query set has 1k vectors
print(f"gist: db {db_gist.shape}, learn {db_gist.shape}, qr {qr_gist.shape}")

Reading File - /data/cpanourg/2-hdvc/data/gist/gist_base.fvecs, with 1000000 vectors:(1000000, 960)
Reading File - /data/cpanourg/2-hdvc/data/gist/gist_learn.fvecs, with 500000 vectors:(500000, 960)
Reading File - /data/cpanourg/2-hdvc/data/gist/gist_query.fvecs, with 1000 vectors:(1000, 960)
gist: db (500000, 960), learn (500000, 960), qr (1000, 960)


In [7]:
# msmarco (.fvecs)
db_msmarco = read_fvecs_cnt(str(DATA_ROOT / "msmarco/base1m.fvecs"), MAX_VECTORS or 1_000_000)
db_msmarco = read_fvecs_cnt(str(DATA_ROOT / "msmarco/train1m.fvecs"), MAX_VECTORS or 1_000_000)
qr_msmarco = read_fvecs_cnt(str(DATA_ROOT / "msmarco/query10k.fvecs"), 10_000)
print(f"msmarco: db {db_msmarco.shape}, train {db_msmarco.shape}, qr {qr_msmarco.shape}")

Reading File - /data/cpanourg/2-hdvc/data/msmarco/base1m.fvecs, with 1000000 vectors:(1000000, 1024)
Reading File - /data/cpanourg/2-hdvc/data/msmarco/train1m.fvecs, with 1000000 vectors:(1000000, 1024)
Reading File - /data/cpanourg/2-hdvc/data/msmarco/query10k.fvecs, with 10000 vectors:(10000, 1024)
msmarco: db (1000000, 1024), train (1000000, 1024), qr (10000, 1024)


In [8]:
# openai (.fvecs)
db_openai = read_fvecs_cnt(str(DATA_ROOT / "openai/openai_base1m.fvecs"), MAX_VECTORS or 1_000_000)
db_openai = read_fvecs_cnt(str(DATA_ROOT / "openai/openai_train1m.fvecs"), MAX_VECTORS or 1_000_000)
qr_openai = read_fvecs_cnt(str(DATA_ROOT / "openai/openai_query10k.fvecs"), 10_000)
print(f"openai: db {db_openai.shape}, train {db_openai.shape}, qr {qr_openai.shape}")

Reading File - /data/cpanourg/2-hdvc/data/openai/openai_base1m.fvecs, with 1000000 vectors:(1000000, 1536)
Reading File - /data/cpanourg/2-hdvc/data/openai/openai_train1m.fvecs, with 1000000 vectors:(1000000, 1536)
Reading File - /data/cpanourg/2-hdvc/data/openai/openai_query10k.fvecs, with 10000 vectors:(10000, 1536)
openai: db (1000000, 1536), train (1000000, 1536), qr (10000, 1536)
